# Notebook 08: Error Analysis and Interpretation

**Author:** Anthony Amit Biswas

## What this notebook does

Analyses frozen evaluation outputs, including false positives, false negatives, boundary errors, category errors, and contextual attributes.


## Step 0 — Environment and Google Drive

This cell mounts Google Drive when the notebook is running in Colab. It is harmless outside Colab.

In [ ]:
from pathlib import Path
import json
import math
import re
import shutil
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
)

from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNNING_IN_COLAB = True
except Exception:
    RUNNING_IN_COLAB = False

print(f"Running in Colab: {RUNNING_IN_COLAB}")
print(f"pandas version: {pd.__version__}")
print(f"numpy version : {np.__version__}")

## Step 1 — Configuration

Adjust only `PROJECT_ROOT` when your Drive folder differs.

Notebook 07 is expected to have written its outputs to:

`/content/drive/MyDrive/Dissertation/outputs/evaluation_lexicon_v2`

In [ ]:
# USER CONFIGURATION

PROJECT_ROOT = Path("/content/drive/MyDrive/Dissertation")

EVALUATION_DIRECTORY = (
    PROJECT_ROOT
    / "outputs"
    / "evaluation_lexicon_v2"
)

ANALYSIS_DIRECTORY = (
    PROJECT_ROOT
    / "outputs"
    / "notebook_08_error_analysis"
)

TABLE_DIRECTORY = ANALYSIS_DIRECTORY / "tables"
FIGURE_DIRECTORY = ANALYSIS_DIRECTORY / "figures"
POSTER_FIGURE_DIRECTORY = ANALYSIS_DIRECTORY / "poster_figures"
PAPER_FIGURE_DIRECTORY = ANALYSIS_DIRECTORY / "paper_figures"
REPORT_DIRECTORY = ANALYSIS_DIRECTORY / "reports"

for directory in [
    ANALYSIS_DIRECTORY,
    TABLE_DIRECTORY,
    FIGURE_DIRECTORY,
    POSTER_FIGURE_DIRECTORY,
    PAPER_FIGURE_DIRECTORY,
    REPORT_DIRECTORY,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Evaluation input directory:")
print(EVALUATION_DIRECTORY)

print("\nNotebook 08 output directory:")
print(ANALYSIS_DIRECTORY)

## Step 2 — Input-file registry

The registry uses the exact filenames generated by the completed Notebook 07.

Some files are essential; others are optional. Optional analyses are skipped safely when their source file is unavailable.

In [ ]:
INPUT_FILES = {
    # Core exact-span evaluation
    "exact_matches": "exact_span_matches.csv",
    "false_positives": "exact_span_false_positives.csv",
    "false_negatives": "exact_span_false_negatives.csv",
    "exact_span_metrics": "exact_span_metrics.csv",

    # Boundary and root-cause diagnostics
    "boundary_overlaps": "boundary_overlap_diagnostic.csv",
    "fn_root_causes": "false_negative_root_cause_analysis.csv",
    "fn_root_cause_summary": "false_negative_root_cause_summary.csv",
    "lexicon_gap_expressions": "lexicon_gap_expressions.csv",
    "lexicon_gap_categories": "lexicon_gap_categories.csv",
    "lexicon_gap_notes": "lexicon_gap_notes.csv",

    # Category analysis
    "category_evaluation": "exact_span_category_classification.csv",
    "category_summary": "exact_span_category_summary.csv",
    "category_report": "exact_span_category_report.csv",
    "category_confusion": "exact_span_category_confusion_matrix.csv",
    "category_errors": "exact_span_category_errors.csv",

    # Assertion analysis
    "assertion_evaluation": "assertion_evaluation_exact_spans.csv",
    "assertion_report": "assertion_classification_report.csv",
    "assertion_confusion": "assertion_confusion_matrix.csv",
    "assertion_errors": "assertion_errors.csv",

    # Temporality analysis
    "temporality_evaluation": "temporality_evaluation_exact_spans.csv",
    "temporality_report": "temporality_classification_report.csv",
    "temporality_confusion": "temporality_confusion_matrix.csv",
    "temporality_errors": "temporality_errors.csv",
    "context_summary": "context_attribute_summary.csv",

    # End-to-end evaluation
    "strict_detail": "strict_exact_match_attribute_detail.csv",
    "strict_metrics": "strict_end_to_end_metrics.csv",
    "relaxed_span_matches": "relaxed_overlap_span_matches.csv",
    "relaxed_category_matches": "relaxed_overlap_category_matches.csv",
    "relaxed_metrics": "relaxed_overlap_metrics.csv",
    "per_category_metrics": "per_category_exact_end_to_end_metrics.csv",
    "per_category_macro": "per_category_exact_macro_summary.csv",

    # Final Notebook 07 outputs
    "final_summary": "notebook_07_final_evaluation_summary.csv",
    "final_validation": "notebook_07_final_validation.csv",
    "evaluation_report": "notebook_07_evaluation_report.json",
}

REQUIRED_KEYS = {
    "exact_matches",
    "false_positives",
    "false_negatives",
    "exact_span_metrics",
    "strict_metrics",
    "relaxed_metrics",
    "per_category_metrics",
    "final_summary",
    "final_validation",
    "evaluation_report",
}

manifest_rows = []

for key, filename in INPUT_FILES.items():
    path = EVALUATION_DIRECTORY / filename
    manifest_rows.append(
        {
            "key": key,
            "filename": filename,
            "required": key in REQUIRED_KEYS,
            "exists": path.exists(),
            "path": str(path),
        }
    )

input_manifest_df = pd.DataFrame(manifest_rows)
display(input_manifest_df)

missing_required = input_manifest_df.loc[
    input_manifest_df["required"]
    & ~input_manifest_df["exists"]
]

if not missing_required.empty:
    missing_names = missing_required["filename"].tolist()
    raise FileNotFoundError(
        "Notebook 08 cannot start because required Notebook 07 files "
        "are missing:\n- "
        + "\n- ".join(missing_names)
        + "\n\nCheck EVALUATION_DIRECTORY and rerun Notebook 07."
    )

input_manifest_df.to_csv(
    TABLE_DIRECTORY / "notebook_08_input_manifest.csv",
    index=False,
)

print("\nAll required Notebook 07 files are available.")

## Step 3 — Utility functions

These helpers make the notebook robust to small differences in column naming and save every figure in PNG, PDF, and SVG formats.

In [ ]:
def load_csv(key, required=False, **kwargs):
    path = EVALUATION_DIRECTORY / INPUT_FILES[key]

    if not path.exists():
        if required:
            raise FileNotFoundError(path)

        print(f"[SKIP] Optional file not found: {path.name}")
        return pd.DataFrame()

    dataframe = pd.read_csv(path, **kwargs)
    print(
        f"[LOAD] {path.name}: "
        f"{len(dataframe):,} rows × {len(dataframe.columns):,} columns"
    )
    return dataframe


def first_existing_column(dataframe, candidates, required=False):
    for column in candidates:
        if column in dataframe.columns:
            return column

    if required:
        raise KeyError(
            "None of the expected columns were found: "
            + ", ".join(candidates)
        )

    return None


def safe_rate(numerator, denominator):
    if denominator == 0:
        return np.nan
    return numerator / denominator


def normalise_label(value):
    if pd.isna(value):
        return "MISSING"

    text = str(value).strip()
    return text if text else "MISSING"


def clean_axis_label(value):
    text = str(value)
    text = re.sub(r"^(GOLD|PRED)::", "", text)
    text = text.replace("_", " ")
    return text


def save_figure(
    figure,
    filename_stem,
    poster_copy=False,
    paper_copy=False,
    dpi=300,
):
    destinations = [FIGURE_DIRECTORY]

    if poster_copy:
        destinations.append(POSTER_FIGURE_DIRECTORY)

    if paper_copy:
        destinations.append(PAPER_FIGURE_DIRECTORY)

    for directory in destinations:
        for extension in ["png", "pdf", "svg"]:
            output_path = directory / f"{filename_stem}.{extension}"

            figure.savefig(
                output_path,
                dpi=dpi,
                bbox_inches="tight",
            )

    print(f"[SAVE] {filename_stem} → PNG, PDF, SVG")


def add_bar_labels(axis, decimals=3):
    for container in axis.containers:
        try:
            labels = []

            for bar in container:
                height = bar.get_height()

                if pd.isna(height):
                    labels.append("")
                elif decimals == 0:
                    labels.append(f"{height:.0f}")
                else:
                    labels.append(f"{height:.{decimals}f}")

            axis.bar_label(
                container,
                labels=labels,
                padding=3,
                fontsize=9,
            )
        except Exception:
            continue


def display_and_save_table(dataframe, filename, index=False):
    display(dataframe)
    dataframe.to_csv(TABLE_DIRECTORY / filename, index=index)
    print(f"[SAVE] {filename}")


def truncated_text(value, max_length=140):
    if pd.isna(value):
        return ""

    text = re.sub(r"\s+", " ", str(value)).strip()

    if len(text) <= max_length:
        return text

    return text[: max_length - 1] + "…"


def infer_count_column(dataframe):
    preferred = [
        "Count",
        "count",
        "support",
        "Support",
        "mentions",
        "Mentions",
        "frequency",
        "Frequency",
    ]

    column = first_existing_column(dataframe, preferred)

    if column is not None:
        return column

    numeric_columns = dataframe.select_dtypes(
        include=[np.number]
    ).columns.tolist()

    return numeric_columns[0] if numeric_columns else None


def prepare_confusion_matrix(dataframe):
    if dataframe.empty:
        return dataframe

    matrix = dataframe.copy()

    first_column = matrix.columns[0]

    if (
        str(first_column).startswith("Unnamed")
        or str(first_column).lower() in {"index", "label"}
    ):
        matrix = matrix.set_index(first_column)

    matrix.index = [clean_axis_label(value) for value in matrix.index]
    matrix.columns = [
        clean_axis_label(value) for value in matrix.columns
    ]

    return matrix.apply(pd.to_numeric, errors="coerce").fillna(0)


def plot_confusion_matrix(
    matrix,
    title,
    filename_stem,
    normalise=False,
):
    if matrix.empty:
        print(f"[SKIP] {title}: empty confusion matrix")
        return

    plot_matrix = matrix.astype(float).copy()

    if normalise:
        row_totals = plot_matrix.sum(axis=1).replace(0, np.nan)
        plot_matrix = plot_matrix.div(row_totals, axis=0).fillna(0)

    width = max(7, 0.65 * len(plot_matrix.columns) + 4)
    height = max(6, 0.55 * len(plot_matrix.index) + 3)

    figure, axis = plt.subplots(figsize=(width, height))

    image = axis.imshow(
        plot_matrix.values,
        aspect="auto",
    )

    axis.set_xticks(np.arange(len(plot_matrix.columns)))
    axis.set_yticks(np.arange(len(plot_matrix.index)))

    axis.set_xticklabels(
        plot_matrix.columns,
        rotation=45,
        ha="right",
    )

    axis.set_yticklabels(plot_matrix.index)

    axis.set_xlabel("Predicted label")
    axis.set_ylabel("Gold label")
    axis.set_title(title)

    threshold = (
        plot_matrix.values.max() / 2
        if plot_matrix.size
        else 0
    )

    for row_index in range(plot_matrix.shape[0]):
        for column_index in range(plot_matrix.shape[1]):
            value = plot_matrix.iloc[row_index, column_index]

            label = (
                f"{value:.2f}"
                if normalise
                else f"{int(round(value))}"
            )

            axis.text(
                column_index,
                row_index,
                label,
                ha="center",
                va="center",
                color=(
                    "white"
                    if value > threshold
                    else "black"
                ),
                fontsize=8,
            )

    figure.colorbar(image, ax=axis)
    figure.tight_layout()

    save_figure(
        figure,
        filename_stem,
        paper_copy=True,
    )

    plt.show()
    plt.close(figure)

## Step 4 — Load Notebook 07 outputs

In [ ]:
data = {}

for key in INPUT_FILES:
    if key == "evaluation_report":
        continue

    data[key] = load_csv(
        key,
        required=(key in REQUIRED_KEYS),
    )

with open(
    EVALUATION_DIRECTORY / INPUT_FILES["evaluation_report"],
    "r",
    encoding="utf-8",
) as report_file:
    evaluation_report = json.load(report_file)

print("\nEvaluation report loaded.")
print(
    json.dumps(
        {
            "corpus": evaluation_report.get("corpus", {}),
            "exact_span_detection":
                evaluation_report.get("exact_span_detection", {}),
            "strict_full_extraction":
                evaluation_report.get("strict_full_extraction", {}),
        },
        indent=2,
    )
)

## Step 5 — Validate the frozen evaluation run

Notebook 08 must stop if Notebook 07's validation report contains any failed check.

In [ ]:
final_validation_df = data["final_validation"].copy()

passed_column = first_existing_column(
    final_validation_df,
    ["Passed", "passed"],
    required=True,
)

passed_values = (
    final_validation_df[passed_column]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({"true": True, "false": False})
)

if passed_values.isna().any():
    passed_values = final_validation_df[passed_column].astype(bool)

failed_checks_df = final_validation_df.loc[
    ~passed_values
].copy()

display(final_validation_df)

if not failed_checks_df.empty:
    raise ValueError(
        "Notebook 07 validation contains failed checks. "
        "Notebook 08 has stopped to protect research integrity."
    )

print("\nNotebook 07 validation confirmed: all checks passed.")

# Part A — Corpus and overall performance

## Step 6 — Corpus summary

In [ ]:
corpus_report = evaluation_report.get("corpus", {})
span_report = evaluation_report.get("exact_span_detection", {})

corpus_summary_df = pd.DataFrame(
    [
        {
            "Measure": "Reviewed notes",
            "Value": corpus_report.get("reviewed_notes", np.nan),
        },
        {
            "Measure": "Gold mentions",
            "Value": corpus_report.get("gold_mentions", np.nan),
        },
        {
            "Measure": "Canonical predictions",
            "Value": corpus_report.get(
                "canonical_predictions",
                np.nan,
            ),
        },
        {
            "Measure": "Exact-span true positives",
            "Value": span_report.get("true_positives", np.nan),
        },
        {
            "Measure": "Exact-span false positives",
            "Value": span_report.get("false_positives", np.nan),
        },
        {
            "Measure": "Exact-span false negatives",
            "Value": span_report.get("false_negatives", np.nan),
        },
    ]
)

display_and_save_table(
    corpus_summary_df,
    "table_01_corpus_summary.csv",
)

figure, axis = plt.subplots(figsize=(9, 5))

plot_df = corpus_summary_df.iloc[:3].copy()

axis.bar(
    plot_df["Measure"],
    plot_df["Value"],
)

axis.set_ylabel("Count")
axis.set_title("Evaluation corpus overview")
axis.tick_params(axis="x", rotation=20)

add_bar_labels(axis, decimals=0)
figure.tight_layout()

save_figure(
    figure,
    "figure_01_corpus_overview",
    poster_copy=True,
    paper_copy=True,
)

plt.show()
plt.close(figure)

## Step 7 — Consolidated performance table

This step reconstructs a clean comparison of the main evaluation levels from the final Notebook 07 report.

In [ ]:
def report_metric(section, metric_name):
    section_data = evaluation_report.get(section, {})

    aliases = {
        "precision": ["precision", "Precision"],
        "recall": ["recall", "Recall"],
        "f1": ["f1", "F1-score", "F1"],
    }

    for key in aliases[metric_name]:
        if key in section_data:
            return float(section_data[key])

    return np.nan


performance_summary_df = pd.DataFrame(
    [
        {
            "Evaluation": "Exact span",
            "Precision": report_metric(
                "exact_span_detection",
                "precision",
            ),
            "Recall": report_metric(
                "exact_span_detection",
                "recall",
            ),
            "F1": report_metric(
                "exact_span_detection",
                "f1",
            ),
        },
        {
            "Evaluation": "Exact span + category",
            "Precision": report_metric(
                "exact_span_category",
                "precision",
            ),
            "Recall": report_metric(
                "exact_span_category",
                "recall",
            ),
            "F1": report_metric(
                "exact_span_category",
                "f1",
            ),
        },
        {
            "Evaluation": "Relaxed overlap + category",
            "Precision": report_metric(
                "relaxed_overlap_category",
                "precision",
            ),
            "Recall": report_metric(
                "relaxed_overlap_category",
                "recall",
            ),
            "F1": report_metric(
                "relaxed_overlap_category",
                "f1",
            ),
        },
        {
            "Evaluation": "Strict full extraction",
            "Precision": report_metric(
                "strict_full_extraction",
                "precision",
            ),
            "Recall": report_metric(
                "strict_full_extraction",
                "recall",
            ),
            "F1": report_metric(
                "strict_full_extraction",
                "f1",
            ),
        },
    ]
)

display_and_save_table(
    performance_summary_df.round(4),
    "table_02_overall_performance.csv",
)

figure, axis = plt.subplots(figsize=(10, 6))

axis.bar(
    performance_summary_df["Evaluation"],
    performance_summary_df["F1"],
)

axis.set_ylim(0, 1.05)
axis.set_ylabel("F1-score")
axis.set_title("Performance across evaluation strictness levels")
axis.tick_params(axis="x", rotation=20)

add_bar_labels(axis, decimals=4)
figure.tight_layout()

save_figure(
    figure,
    "figure_02_overall_f1_comparison",
    poster_copy=True,
    paper_copy=True,
)

plt.show()
plt.close(figure)

## Step 8 — Precision, recall, and F1 comparison

In [ ]:
metric_plot_df = performance_summary_df.melt(
    id_vars="Evaluation",
    value_vars=["Precision", "Recall", "F1"],
    var_name="Metric",
    value_name="Score",
)

evaluation_names = performance_summary_df["Evaluation"].tolist()
metric_names = ["Precision", "Recall", "F1"]

x_positions = np.arange(len(evaluation_names))
bar_width = 0.24

figure, axis = plt.subplots(figsize=(12, 6))

for metric_index, metric_name in enumerate(metric_names):
    values = (
        performance_summary_df
        .set_index("Evaluation")
        .loc[evaluation_names, metric_name]
        .values
    )

    axis.bar(
        x_positions
        + (metric_index - 1) * bar_width,
        values,
        width=bar_width,
        label=metric_name,
    )

axis.set_xticks(x_positions)
axis.set_xticklabels(evaluation_names, rotation=18, ha="right")
axis.set_ylim(0, 1.05)
axis.set_ylabel("Score")
axis.set_title("Precision, recall, and F1 by evaluation level")
axis.legend()

figure.tight_layout()

save_figure(
    figure,
    "figure_03_precision_recall_f1_comparison",
    paper_copy=True,
)

plt.show()
plt.close(figure)

# Part B — False-positive and false-negative analysis

## Step 9 — False-positive profile

In [ ]:
false_positive_df = data["false_positives"].copy()

fp_category_column = first_existing_column(
    false_positive_df,
    [
        "complication_category",
        "pred_category",
        "category",
        "predicted_category",
    ],
)

fp_assertion_column = first_existing_column(
    false_positive_df,
    [
        "predicted_assertion",
        "assertion",
        "prediction_assertion",
    ],
)

fp_temporality_column = first_existing_column(
    false_positive_df,
    [
        "predicted_temporality",
        "temporality",
        "prediction_temporality",
    ],
)

fp_note_column = first_existing_column(
    false_positive_df,
    ["note_id", "NOTE_ID", "document_id"],
)

fp_breakdowns = {}

for name, column in {
    "category": fp_category_column,
    "assertion": fp_assertion_column,
    "temporality": fp_temporality_column,
}.items():
    if column is None:
        continue

    summary_df = (
        false_positive_df[column]
        .map(normalise_label)
        .value_counts(dropna=False)
        .rename_axis(name)
        .reset_index(name="Count")
    )

    summary_df["Percentage"] = (
        100
        * summary_df["Count"]
        / max(len(false_positive_df), 1)
    ).round(2)

    fp_breakdowns[name] = summary_df

    display_and_save_table(
        summary_df,
        f"table_03_fp_by_{name}.csv",
    )

if fp_note_column is not None:
    fp_by_note_df = (
        false_positive_df.groupby(fp_note_column)
        .size()
        .sort_values(ascending=False)
        .rename("False positives")
        .reset_index()
    )

    display_and_save_table(
        fp_by_note_df,
        "table_03_fp_by_note.csv",
    )

if "category" in fp_breakdowns:
    fp_category_plot_df = fp_breakdowns["category"].head(15)

    figure, axis = plt.subplots(figsize=(10, 7))

    axis.barh(
        fp_category_plot_df["category"][::-1],
        fp_category_plot_df["Count"][::-1],
    )

    axis.set_xlabel("False-positive count")
    axis.set_ylabel("Predicted complication category")
    axis.set_title("Most frequent false-positive categories")

    figure.tight_layout()

    save_figure(
        figure,
        "figure_04_false_positives_by_category",
        poster_copy=True,
        paper_copy=True,
    )

    plt.show()
    plt.close(figure)

## Step 10 — False-negative profile and root causes

In [ ]:
false_negative_df = data["false_negatives"].copy()
fn_rootcause_df = data["fn_root_causes"].copy()
fn_rootcause_summary_df = data["fn_root_cause_summary"].copy()

fn_category_column = first_existing_column(
    false_negative_df,
    [
        "gold_category",
        "complication_category",
        "category",
    ],
)

fn_note_column = first_existing_column(
    false_negative_df,
    ["note_id", "NOTE_ID", "document_id"],
)

if fn_category_column is not None:
    fn_by_category_df = (
        false_negative_df[fn_category_column]
        .map(normalise_label)
        .value_counts(dropna=False)
        .rename_axis("category")
        .reset_index(name="Count")
    )

    fn_by_category_df["Percentage"] = (
        100
        * fn_by_category_df["Count"]
        / max(len(false_negative_df), 1)
    ).round(2)

    display_and_save_table(
        fn_by_category_df,
        "table_04_fn_by_category.csv",
    )

    figure, axis = plt.subplots(figsize=(10, 7))

    fn_plot_df = fn_by_category_df.head(15)

    axis.barh(
        fn_plot_df["category"][::-1],
        fn_plot_df["Count"][::-1],
    )

    axis.set_xlabel("False-negative count")
    axis.set_ylabel("Gold complication category")
    axis.set_title("Most frequently missed complication categories")

    figure.tight_layout()

    save_figure(
        figure,
        "figure_05_false_negatives_by_category",
        poster_copy=True,
        paper_copy=True,
    )

    plt.show()
    plt.close(figure)

if fn_note_column is not None:
    fn_by_note_df = (
        false_negative_df.groupby(fn_note_column)
        .size()
        .sort_values(ascending=False)
        .rename("False negatives")
        .reset_index()
    )

    display_and_save_table(
        fn_by_note_df,
        "table_04_fn_by_note.csv",
    )

if not fn_rootcause_summary_df.empty:
    display_and_save_table(
        fn_rootcause_summary_df,
        "table_04_false_negative_root_causes.csv",
    )

    root_count_column = infer_count_column(
        fn_rootcause_summary_df
    )

    root_label_column = first_existing_column(
        fn_rootcause_summary_df,
        [
            "Cause",
            "cause",
            "root_cause",
            "Root Cause",
            "classification",
            "Category",
            "category",
        ],
    )

    if root_label_column and root_count_column:
        root_plot_df = (
            fn_rootcause_summary_df[
                [root_label_column, root_count_column]
            ]
            .copy()
            .sort_values(root_count_column)
        )

        figure, axis = plt.subplots(figsize=(9, 5))

        axis.barh(
            root_plot_df[root_label_column],
            root_plot_df[root_count_column],
        )

        axis.set_xlabel("False-negative count")
        axis.set_ylabel("Root-cause classification")
        axis.set_title("False-negative root-cause analysis")

        for container in axis.containers:
            axis.bar_label(container, padding=3, fontsize=9)

        figure.tight_layout()

        save_figure(
            figure,
            "figure_06_false_negative_root_causes",
            poster_copy=True,
            paper_copy=True,
        )

        plt.show()
        plt.close(figure)
    else:
        print(
            "[WARNING] False-negative root-cause table was loaded, "
            "but its label/count columns could not be identified."
        )

# Part C — Boundary, category, assertion, and temporality analysis

## Step 11 — Boundary-overlap diagnostics

In [ ]:
boundary_df = data["boundary_overlaps"].copy()

if boundary_df.empty:
    print("No boundary-overlap diagnostic rows were available.")
else:
    iou_column = first_existing_column(
        boundary_df,
        [
            "iou",
            "IoU",
            "span_iou",
            "overlap_iou",
        ],
    )

    gold_start_column = first_existing_column(
        boundary_df,
        ["gold_start", "gold_start_char", "annotation_start"],
    )

    gold_end_column = first_existing_column(
        boundary_df,
        ["gold_end", "gold_end_char", "annotation_end"],
    )

    pred_start_column = first_existing_column(
        boundary_df,
        ["pred_start", "start_char", "prediction_start"],
    )

    pred_end_column = first_existing_column(
        boundary_df,
        ["pred_end", "end_char", "prediction_end"],
    )

    if iou_column is None and all(
        column is not None
        for column in [
            gold_start_column,
            gold_end_column,
            pred_start_column,
            pred_end_column,
        ]
    ):
        intersection = (
            np.minimum(
                boundary_df[gold_end_column],
                boundary_df[pred_end_column],
            )
            - np.maximum(
                boundary_df[gold_start_column],
                boundary_df[pred_start_column],
            )
        ).clip(lower=0)

        union = (
            np.maximum(
                boundary_df[gold_end_column],
                boundary_df[pred_end_column],
            )
            - np.minimum(
                boundary_df[gold_start_column],
                boundary_df[pred_start_column],
            )
        )

        boundary_df["computed_iou"] = (
            intersection / union.replace(0, np.nan)
        )

        iou_column = "computed_iou"

    if all(
        column is not None
        for column in [
            gold_start_column,
            gold_end_column,
            pred_start_column,
            pred_end_column,
        ]
    ):
        boundary_df["start_offset_difference"] = (
            boundary_df[pred_start_column]
            - boundary_df[gold_start_column]
        )

        boundary_df["end_offset_difference"] = (
            boundary_df[pred_end_column]
            - boundary_df[gold_end_column]
        )

        boundary_df["absolute_boundary_error"] = (
            boundary_df["start_offset_difference"].abs()
            + boundary_df["end_offset_difference"].abs()
        )

    boundary_summary_rows = [
        {
            "Metric": "Boundary-overlap pairs",
            "Value": len(boundary_df),
        }
    ]

    if iou_column is not None:
        boundary_summary_rows.extend(
            [
                {
                    "Metric": "Mean IoU",
                    "Value": boundary_df[iou_column].mean(),
                },
                {
                    "Metric": "Median IoU",
                    "Value": boundary_df[iou_column].median(),
                },
                {
                    "Metric": "Minimum IoU",
                    "Value": boundary_df[iou_column].min(),
                },
                {
                    "Metric": "Maximum IoU",
                    "Value": boundary_df[iou_column].max(),
                },
            ]
        )

    if "absolute_boundary_error" in boundary_df.columns:
        boundary_summary_rows.extend(
            [
                {
                    "Metric": "Mean absolute boundary error",
                    "Value":
                        boundary_df["absolute_boundary_error"].mean(),
                },
                {
                    "Metric": "Median absolute boundary error",
                    "Value":
                        boundary_df["absolute_boundary_error"].median(),
                },
            ]
        )

    boundary_summary_df = pd.DataFrame(boundary_summary_rows)

    display_and_save_table(
        boundary_summary_df.round(4),
        "table_05_boundary_summary.csv",
    )

    boundary_df.to_csv(
        TABLE_DIRECTORY / "table_05_boundary_pairs_enriched.csv",
        index=False,
    )

    if iou_column is not None:
        figure, axis = plt.subplots(figsize=(8, 5))

        axis.hist(
            boundary_df[iou_column].dropna(),
            bins=min(10, max(4, len(boundary_df))),
            edgecolor="black",
        )

        axis.set_xlabel("Intersection over Union")
        axis.set_ylabel("Number of overlap pairs")
        axis.set_title("Boundary-overlap similarity distribution")

        figure.tight_layout()

        save_figure(
            figure,
            "figure_07_boundary_iou_distribution",
            paper_copy=True,
        )

        plt.show()
        plt.close(figure)

## Step 12 — Category classification

In [ ]:
category_summary_df = data["category_summary"].copy()
category_report_df = data["category_report"].copy()
category_errors_df = data["category_errors"].copy()

display_and_save_table(
    category_summary_df,
    "table_06_category_classification_summary.csv",
)

display_and_save_table(
    category_report_df,
    "table_06_category_classification_report.csv",
)

category_confusion_df = prepare_confusion_matrix(
    data["category_confusion"]
)

plot_confusion_matrix(
    category_confusion_df,
    "Complication-category confusion matrix",
    "figure_08_category_confusion_matrix",
    normalise=False,
)

plot_confusion_matrix(
    category_confusion_df,
    "Row-normalised complication-category confusion matrix",
    "figure_09_category_confusion_matrix_normalised",
    normalise=True,
)

if not category_errors_df.empty:
    gold_category_column = first_existing_column(
        category_errors_df,
        ["gold_category", "gold_label"],
    )

    predicted_category_column = first_existing_column(
        category_errors_df,
        ["pred_category", "complication_category", "predicted_label"],
    )

    if gold_category_column and predicted_category_column:
        category_error_pairs_df = (
            category_errors_df.groupby(
                [
                    gold_category_column,
                    predicted_category_column,
                ],
                dropna=False,
            )
            .size()
            .sort_values(ascending=False)
            .rename("Count")
            .reset_index()
        )

        display_and_save_table(
            category_error_pairs_df,
            "table_06_category_error_pairs.csv",
        )

## Step 13 — Assertion classification

The notebook reports both:

- the conventional full-label metrics, including labels with zero gold support;
- support-aware macro metrics calculated only across labels present in the gold standard.

This prevents rare or unsupported classes from being hidden by weighted averages while also avoiding an uninterpretable macro average dominated by zero-support labels.

In [ ]:
assertion_summary_df = data["context_summary"].copy()

if not assertion_summary_df.empty:
    assertion_name_column = first_existing_column(
        assertion_summary_df,
        ["Attribute", "attribute"],
    )

    if assertion_name_column is not None:
        assertion_only_summary_df = assertion_summary_df.loc[
            assertion_summary_df[assertion_name_column]
            .astype(str)
            .str.upper()
            .eq("ASSERTION")
        ].copy()
    else:
        assertion_only_summary_df = pd.DataFrame()
else:
    assertion_only_summary_df = pd.DataFrame()

display_and_save_table(
    assertion_only_summary_df,
    "table_07_assertion_summary.csv",
)

display_and_save_table(
    data["assertion_report"],
    "table_07_assertion_classification_report.csv",
)

assertion_evaluation_df = data["assertion_evaluation"].copy()

assertion_gold_column = first_existing_column(
    assertion_evaluation_df,
    ["gold_label", "gold_assertion", "gold"],
    required=True,
)

assertion_pred_column = first_existing_column(
    assertion_evaluation_df,
    ["predicted_label", "pred_assertion", "prediction"],
    required=True,
)

assertion_gold = (
    assertion_evaluation_df[assertion_gold_column]
    .map(normalise_label)
)

assertion_pred = (
    assertion_evaluation_df[assertion_pred_column]
    .map(normalise_label)
)

assertion_supported_labels = sorted(
    assertion_gold.value_counts()
    .loc[lambda series: series > 0]
    .index
    .tolist()
)

assertion_supported_precision, assertion_supported_recall, assertion_supported_f1, _ = (
    precision_recall_fscore_support(
        assertion_gold,
        assertion_pred,
        labels=assertion_supported_labels,
        average="macro",
        zero_division=0,
    )
)

assertion_weighted_precision, assertion_weighted_recall, assertion_weighted_f1, _ = (
    precision_recall_fscore_support(
        assertion_gold,
        assertion_pred,
        average="weighted",
        zero_division=0,
    )
)

assertion_research_summary_df = pd.DataFrame(
    [
        {
            "Metric": "Accuracy",
            "Value": accuracy_score(
                assertion_gold,
                assertion_pred,
            ),
            "Scope": "All exact-span matches",
        },
        {
            "Metric": "Support-aware macro precision",
            "Value": assertion_supported_precision,
            "Scope": (
                "Gold-supported labels only: "
                + ", ".join(assertion_supported_labels)
            ),
        },
        {
            "Metric": "Support-aware macro recall",
            "Value": assertion_supported_recall,
            "Scope": (
                "Gold-supported labels only: "
                + ", ".join(assertion_supported_labels)
            ),
        },
        {
            "Metric": "Support-aware macro F1",
            "Value": assertion_supported_f1,
            "Scope": (
                "Gold-supported labels only: "
                + ", ".join(assertion_supported_labels)
            ),
        },
        {
            "Metric": "Weighted F1",
            "Value": assertion_weighted_f1,
            "Scope": "All gold instances",
        },
    ]
)

display_and_save_table(
    assertion_research_summary_df.round(4),
    "table_07_assertion_research_summary.csv",
)

assertion_confusion_df = prepare_confusion_matrix(
    data["assertion_confusion"]
)

plot_confusion_matrix(
    assertion_confusion_df,
    "Assertion-status confusion matrix",
    "figure_10_assertion_confusion_matrix",
    normalise=False,
)

plot_confusion_matrix(
    assertion_confusion_df,
    "Row-normalised assertion-status confusion matrix",
    "figure_11_assertion_confusion_matrix_normalised",
    normalise=True,
)

assertion_errors_df = data["assertion_errors"].copy()

if not assertion_errors_df.empty:
    assertion_pair_df = (
        assertion_errors_df.groupby(
            ["gold_label", "predicted_label"],
            dropna=False,
        )
        .size()
        .sort_values(ascending=False)
        .rename("Count")
        .reset_index()
    )

    display_and_save_table(
        assertion_pair_df,
        "table_07_assertion_error_pairs.csv",
    )

## Step 14 — Temporality classification

As with assertion, both conventional and support-aware metrics are retained. The support-aware macro result includes only temporality classes represented in the gold standard.

In [ ]:
if not assertion_summary_df.empty:
    assertion_name_column = first_existing_column(
        assertion_summary_df,
        ["Attribute", "attribute"],
    )

    if assertion_name_column is not None:
        temporality_only_summary_df = assertion_summary_df.loc[
            assertion_summary_df[assertion_name_column]
            .astype(str)
            .str.upper()
            .eq("TEMPORALITY")
        ].copy()
    else:
        temporality_only_summary_df = pd.DataFrame()
else:
    temporality_only_summary_df = pd.DataFrame()

display_and_save_table(
    temporality_only_summary_df,
    "table_08_temporality_summary.csv",
)

display_and_save_table(
    data["temporality_report"],
    "table_08_temporality_classification_report.csv",
)

temporality_evaluation_df = data["temporality_evaluation"].copy()

temporality_gold_column = first_existing_column(
    temporality_evaluation_df,
    ["gold_label", "gold_temporality", "gold"],
    required=True,
)

temporality_pred_column = first_existing_column(
    temporality_evaluation_df,
    ["predicted_label", "pred_temporality", "prediction"],
    required=True,
)

temporality_gold = (
    temporality_evaluation_df[temporality_gold_column]
    .map(normalise_label)
)

temporality_pred = (
    temporality_evaluation_df[temporality_pred_column]
    .map(normalise_label)
)

temporality_supported_labels = sorted(
    temporality_gold.value_counts()
    .loc[lambda series: series > 0]
    .index
    .tolist()
)

temporality_supported_precision, temporality_supported_recall, temporality_supported_f1, _ = (
    precision_recall_fscore_support(
        temporality_gold,
        temporality_pred,
        labels=temporality_supported_labels,
        average="macro",
        zero_division=0,
    )
)

temporality_weighted_precision, temporality_weighted_recall, temporality_weighted_f1, _ = (
    precision_recall_fscore_support(
        temporality_gold,
        temporality_pred,
        average="weighted",
        zero_division=0,
    )
)

temporality_research_summary_df = pd.DataFrame(
    [
        {
            "Metric": "Accuracy",
            "Value": accuracy_score(
                temporality_gold,
                temporality_pred,
            ),
            "Scope": "All exact-span matches",
        },
        {
            "Metric": "Support-aware macro precision",
            "Value": temporality_supported_precision,
            "Scope": (
                "Gold-supported labels only: "
                + ", ".join(temporality_supported_labels)
            ),
        },
        {
            "Metric": "Support-aware macro recall",
            "Value": temporality_supported_recall,
            "Scope": (
                "Gold-supported labels only: "
                + ", ".join(temporality_supported_labels)
            ),
        },
        {
            "Metric": "Support-aware macro F1",
            "Value": temporality_supported_f1,
            "Scope": (
                "Gold-supported labels only: "
                + ", ".join(temporality_supported_labels)
            ),
        },
        {
            "Metric": "Weighted F1",
            "Value": temporality_weighted_f1,
            "Scope": "All gold instances",
        },
    ]
)

display_and_save_table(
    temporality_research_summary_df.round(4),
    "table_08_temporality_research_summary.csv",
)

temporality_confusion_df = prepare_confusion_matrix(
    data["temporality_confusion"]
)

plot_confusion_matrix(
    temporality_confusion_df,
    "Temporality confusion matrix",
    "figure_12_temporality_confusion_matrix",
    normalise=False,
)

plot_confusion_matrix(
    temporality_confusion_df,
    "Row-normalised temporality confusion matrix",
    "figure_13_temporality_confusion_matrix_normalised",
    normalise=True,
)

temporality_errors_df = data["temporality_errors"].copy()

if not temporality_errors_df.empty:
    temporality_pair_df = (
        temporality_errors_df.groupby(
            ["gold_label", "predicted_label"],
            dropna=False,
        )
        .size()
        .sort_values(ascending=False)
        .rename("Count")
        .reset_index()
    )

    display_and_save_table(
        temporality_pair_df,
        "table_08_temporality_error_pairs.csv",
    )

# Part D — Per-category exact-span and category performance

## Step 15 — Ranked per-category exact-span and category metrics

These metrics require:

1. an exact span match; and
2. the correct complication category.

They do **not** include assertion or temporality and must not be described as strict full end-to-end extraction.

In [ ]:
per_category_df = data["per_category_metrics"].copy()

category_column = first_existing_column(
    per_category_df,
    [
        "Category",
        "category",
        "gold_category",
        "complication_category",
    ],
    required=True,
)

f1_column = first_existing_column(
    per_category_df,
    [
        "F1-score",
        "F1",
        "f1",
        "f1_score",
    ],
    required=True,
)

precision_column = first_existing_column(
    per_category_df,
    ["Precision", "precision"],
)

recall_column = first_existing_column(
    per_category_df,
    ["Recall", "recall"],
)

support_column = first_existing_column(
    per_category_df,
    [
        "Gold Mentions",
        "Gold mentions",
        "gold_mentions",
        "Support",
        "support",
        "Gold Count",
    ],
)

per_category_ranked_df = (
    per_category_df
    .sort_values(
        [f1_column, category_column],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

display_and_save_table(
    per_category_ranked_df,
    "table_09_per_category_exact_span_category_metrics.csv",
)

figure_height = max(
    6,
    0.36 * len(per_category_ranked_df) + 2,
)

figure, axis = plt.subplots(
    figsize=(10, figure_height)
)

plot_df = per_category_ranked_df.sort_values(f1_column)

axis.barh(
    plot_df[category_column],
    plot_df[f1_column],
)

axis.set_xlim(0, 1.05)
axis.set_xlabel("Exact span + category F1-score")
axis.set_ylabel("Complication category")
axis.set_title("Per-category exact-span and category performance")

figure.tight_layout()

save_figure(
    figure,
    "figure_14_per_category_exact_span_category_f1",
    poster_copy=True,
    paper_copy=True,
)

plt.show()
plt.close(figure)

## Step 16 — Gold support versus category F1

This analysis shows whether category-level performance is associated with the number of gold-standard mentions. It is exploratory because several categories have very small support.

In [ ]:
if support_column is None:
    raise KeyError(
        "A gold-support column was not found in the per-category metrics. "
        "Expected a column such as 'Gold Mentions' or 'support'."
    )
else:
    support_plot_df = per_category_ranked_df[
        [category_column, support_column, f1_column]
    ].copy()

    support_plot_df[support_column] = pd.to_numeric(
        support_plot_df[support_column],
        errors="coerce",
    )

    support_plot_df[f1_column] = pd.to_numeric(
        support_plot_df[f1_column],
        errors="coerce",
    )

    support_plot_df = support_plot_df.dropna()

    display_and_save_table(
        support_plot_df.sort_values(
            support_column,
            ascending=False,
        ),
        "table_09_category_support_vs_f1.csv",
    )

    figure, axis = plt.subplots(figsize=(9, 6))

    axis.scatter(
        support_plot_df[support_column],
        support_plot_df[f1_column],
        s=55,
        alpha=0.8,
    )

    for _, row in support_plot_df.iterrows():
        axis.annotate(
            str(row[category_column]).replace("_", " "),
            (
                row[support_column],
                row[f1_column],
            ),
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=7,
            alpha=0.85,
        )

    axis.set_xlabel("Gold-standard mentions")
    axis.set_ylabel("Exact span + category F1-score")
    axis.set_ylim(0, 1.05)
    axis.set_title(
        "Gold-standard support versus category performance"
    )

    figure.tight_layout()

    save_figure(
        figure,
        "figure_15_category_support_vs_f1",
        paper_copy=True,
    )

    plt.show()
    plt.close(figure)

# Part E — Integrated error taxonomy

## Step 17 — Build a non-overlapping primary error taxonomy

This taxonomy separates errors by the first evaluation stage at which they fail:

1. exact-span false positive;
2. exact-span false negative;
3. exact-span category error;
4. assertion error after correct span and category;
5. temporality error after correct span, category, and assertion.

This ordering avoids double-counting the same exact-span match in multiple contextual categories.

In [ ]:
exact_matches_df = data["exact_matches"].copy()
category_evaluation_df = data["category_evaluation"].copy()

taxonomy_rows = [
    {
        "Primary error type": "Exact-span false positive",
        "Count": len(false_positive_df),
    },
    {
        "Primary error type": "Exact-span false negative",
        "Count": len(false_negative_df),
    },
    {
        "Primary error type": "Category error on exact span",
        "Count": len(category_errors_df),
    },
]

strict_detail_df = data["strict_detail"].copy()

category_correct_column = first_existing_column(
    strict_detail_df,
    ["category_correct"],
)

assertion_correct_column = first_existing_column(
    strict_detail_df,
    ["assertion_correct"],
)

temporality_correct_column = first_existing_column(
    strict_detail_df,
    ["temporality_correct"],
)

if all(
    column is not None
    for column in [
        category_correct_column,
        assertion_correct_column,
        temporality_correct_column,
    ]
):
    category_correct = (
        strict_detail_df[category_correct_column]
        .astype(str)
        .str.lower()
        .map({"true": True, "false": False})
        .fillna(
            strict_detail_df[category_correct_column].astype(bool)
        )
    )

    assertion_correct = (
        strict_detail_df[assertion_correct_column]
        .astype(str)
        .str.lower()
        .map({"true": True, "false": False})
        .fillna(
            strict_detail_df[assertion_correct_column].astype(bool)
        )
    )

    temporality_correct = (
        strict_detail_df[temporality_correct_column]
        .astype(str)
        .str.lower()
        .map({"true": True, "false": False})
        .fillna(
            strict_detail_df[temporality_correct_column].astype(bool)
        )
    )

    assertion_primary_count = int(
        (
            category_correct
            & ~assertion_correct
        ).sum()
    )

    temporality_primary_count = int(
        (
            category_correct
            & assertion_correct
            & ~temporality_correct
        ).sum()
    )
else:
    assertion_primary_count = len(assertion_errors_df)
    temporality_primary_count = len(temporality_errors_df)

taxonomy_rows.extend(
    [
        {
            "Primary error type":
                "Assertion error after correct span/category",
            "Count": assertion_primary_count,
        },
        {
            "Primary error type":
                "Temporality error after correct span/category/assertion",
            "Count": temporality_primary_count,
        },
    ]
)

error_taxonomy_df = pd.DataFrame(taxonomy_rows)

error_taxonomy_df["Percentage"] = (
    100
    * error_taxonomy_df["Count"]
    / max(error_taxonomy_df["Count"].sum(), 1)
).round(2)

display_and_save_table(
    error_taxonomy_df,
    "table_10_primary_error_taxonomy.csv",
)

figure, axis = plt.subplots(figsize=(10, 6))

plot_df = error_taxonomy_df.sort_values("Count")

axis.barh(
    plot_df["Primary error type"],
    plot_df["Count"],
)

axis.set_xlabel("Count")
axis.set_ylabel("Primary error type")
axis.set_title("Primary error taxonomy")

figure.tight_layout()

save_figure(
    figure,
    "figure_16_primary_error_taxonomy",
    poster_copy=True,
    paper_copy=True,
)

plt.show()
plt.close(figure)

## Step 18 — Observed error counts by evaluation dimension

These counts are **not additive**. A single matched mention may have errors in more than one contextual dimension, particularly assertion and temporality. Figure 17 is therefore diagnostic rather than a count of unique errors.

In [ ]:
error_stage_df = pd.DataFrame(
    [
        {
            "Evaluation dimension": "Span detection",
            "Observed errors": (
                len(false_positive_df)
                + len(false_negative_df)
            ),
            "Counting rule":
                "False positives + false negatives",
        },
        {
            "Evaluation dimension": "Category classification",
            "Observed errors": len(category_errors_df),
            "Counting rule":
                "Incorrect category among exact-span matches",
        },
        {
            "Evaluation dimension": "Assertion classification",
            "Observed errors": len(assertion_errors_df),
            "Counting rule":
                "Incorrect assertion among exact-span matches",
        },
        {
            "Evaluation dimension": "Temporality classification",
            "Observed errors": len(temporality_errors_df),
            "Counting rule":
                "Incorrect temporality among exact-span matches",
        },
    ]
)

display_and_save_table(
    error_stage_df,
    "table_11_observed_errors_by_dimension_non_additive.csv",
)

figure, axis = plt.subplots(figsize=(10, 5.5))

axis.bar(
    error_stage_df["Evaluation dimension"],
    error_stage_df["Observed errors"],
)

axis.set_ylabel("Observed error count")
axis.set_title(
    "Observed errors by evaluation dimension "
    "(counts are not additive)"
)
axis.tick_params(axis="x", rotation=15)

add_bar_labels(axis, decimals=0)
figure.tight_layout()

save_figure(
    figure,
    "figure_17_observed_errors_by_dimension_non_additive",
    paper_copy=True,
)

plt.show()
plt.close(figure)

# Part F — Representative examples for qualitative analysis

## Step 19 — Prepare representative error tables

These tables are intended for internal dissertation analysis. Text is whitespace-normalised and truncated to reduce unnecessary exposure of long clinical passages.

Do not publish any full note text or identifiers.

In [ ]:
def representative_examples(
    dataframe,
    example_type,
    maximum_rows=10,
):
    if dataframe.empty:
        return pd.DataFrame()

    output_df = dataframe.copy().head(maximum_rows)

    text_columns = [
        column
        for column in output_df.columns
        if any(
            token in column.lower()
            for token in [
                "text",
                "expression",
                "mention",
                "context",
            ]
        )
    ]

    for column in text_columns:
        output_df[column] = output_df[column].map(
            lambda value: truncated_text(
                value,
                max_length=160,
            )
        )

    identifier_columns = [
        column
        for column in output_df.columns
        if any(
            token in column.lower()
            for token in [
                "subject_id",
                "hadm_id",
                "stay_id",
            ]
        )
    ]

    output_df = output_df.drop(
        columns=identifier_columns,
        errors="ignore",
    )

    output_df.insert(
        0,
        "example_type",
        example_type,
    )

    return output_df


representative_tables = {
    "false_positive_examples":
        representative_examples(
            false_positive_df,
            "False positive",
        ),

    "false_negative_examples":
        representative_examples(
            false_negative_df,
            "False negative",
        ),

    "boundary_examples":
        representative_examples(
            boundary_df,
            "Boundary mismatch",
        ),

    "category_error_examples":
        representative_examples(
            category_errors_df,
            "Category error",
        ),

    "assertion_error_examples":
        representative_examples(
            assertion_errors_df,
            "Assertion error",
        ),

    "temporality_error_examples":
        representative_examples(
            temporality_errors_df,
            "Temporality error",
        ),
}

for name, example_df in representative_tables.items():
    if example_df.empty:
        print(f"[SKIP] {name}: no rows")
        continue

    display_and_save_table(
        example_df,
        f"table_12_{name}.csv",
    )

# Part G — Research-ready summary

## Step 20 — Automatically derive key interpretation statements

The statements below are deliberately conservative. In particular, relaxed matching is interpreted as evidence of partial overlap or boundary disagreement—not proof that every boundary error is minor.

In [ ]:
performance_lookup = (
    performance_summary_df
    .set_index("Evaluation")
    ["F1"]
    .to_dict()
)

exact_f1 = performance_lookup.get("Exact span", np.nan)
exact_category_f1 = performance_lookup.get(
    "Exact span + category",
    np.nan,
)
relaxed_category_f1 = performance_lookup.get(
    "Relaxed overlap + category",
    np.nan,
)
strict_full_f1 = performance_lookup.get(
    "Strict full extraction",
    np.nan,
)

interpretation_rows = [
    {
        "Finding": "Exact-span extraction performance",
        "Value": exact_f1,
        "Interpretation":
            "The system achieved strong strict-boundary mention "
            "detection across the reviewed discharge summaries.",
    },
    {
        "Finding": "Category penalty after exact matching",
        "Value": exact_f1 - exact_category_f1,
        "Interpretation":
            "The small reduction estimates the additional penalty "
            "from category misclassification after exact span detection.",
    },
    {
        "Finding": "Gain from relaxed boundary matching",
        "Value": relaxed_category_f1 - exact_category_f1,
        "Interpretation":
            "Relaxed matching increased F1, indicating that a "
            "proportion of strict errors resulted from partial span "
            "overlap or boundary disagreement.",
    },
    {
        "Finding": "Contextual-attribute penalty",
        "Value": exact_category_f1 - strict_full_f1,
        "Interpretation":
            "The reduction to strict full extraction reflects the "
            "combined burden of assertion and temporality errors.",
    },
]

interpretation_summary_df = pd.DataFrame(
    interpretation_rows
)

display_and_save_table(
    interpretation_summary_df.round(4),
    "table_13_interpretation_summary.csv",
)

## Step 21 — Poster shortlist and export consistency

Only figures that remain legible at poster scale are prioritised. Dense 23-class confusion matrices are retained for the dissertation or appendix rather than the main poster.

In [ ]:
poster_results_df = pd.DataFrame(
    [
        {
            "Poster item": "Reviewed notes",
            "Value": str(
                corpus_report.get("reviewed_notes", "NA")
            ),
        },
        {
            "Poster item": "Gold mentions",
            "Value": str(
                corpus_report.get("gold_mentions", "NA")
            ),
        },
        {
            "Poster item": "System predictions",
            "Value": str(
                corpus_report.get(
                    "canonical_predictions",
                    "NA",
                )
            ),
        },
        {
            "Poster item": "Exact-span F1",
            "Value": f"{exact_f1:.4f}",
        },
        {
            "Poster item": "Exact span + category F1",
            "Value": f"{exact_category_f1:.4f}",
        },
        {
            "Poster item": "Relaxed overlap + category F1",
            "Value": f"{relaxed_category_f1:.4f}",
        },
        {
            "Poster item": "Strict full extraction F1",
            "Value": f"{strict_full_f1:.4f}",
        },
        {
            "Poster item": "Exact-span false positives",
            "Value": str(len(false_positive_df)),
        },
        {
            "Poster item": "Exact-span false negatives",
            "Value": str(len(false_negative_df)),
        },
        {
            "Poster item": "Boundary-overlap pairs",
            "Value": str(len(boundary_df)),
        },
    ]
)

display_and_save_table(
    poster_results_df,
    "table_14_poster_results_shortlist.csv",
)

poster_figure_manifest_df = pd.DataFrame(
    [
        {
            "Priority": 1,
            "Figure":
                "figure_02_overall_f1_comparison",
            "Poster purpose":
                "Headline comparison of evaluation strictness",
        },
        {
            "Priority": 2,
            "Figure":
                "figure_14_per_category_exact_span_category_f1",
            "Poster purpose":
                "Shows variation across complication categories",
        },
        {
            "Priority": 3,
            "Figure":
                "figure_16_primary_error_taxonomy",
            "Poster purpose":
                "Summarises mutually exclusive primary error types",
        },
        {
            "Priority": 4,
            "Figure":
                "figure_06_false_negative_root_causes",
            "Poster purpose":
                "Distinguishes genuine lexicon gaps from note-level misses",
        },
        {
            "Priority": 5,
            "Figure":
                "figure_11_assertion_confusion_matrix_normalised",
            "Poster purpose":
                "Shows contextual assertion performance",
        },
        {
            "Priority": 6,
            "Figure":
                "figure_13_temporality_confusion_matrix_normalised",
            "Poster purpose":
                "Shows contextual temporality performance",
        },
    ]
)

display_and_save_table(
    poster_figure_manifest_df,
    "table_15_poster_figure_manifest.csv",
)

# Ensure every figure in the poster manifest is present in the poster folder.
for figure_stem in poster_figure_manifest_df["Figure"]:
    for extension in ["png", "pdf", "svg"]:
        source_file = FIGURE_DIRECTORY / f"{figure_stem}.{extension}"
        destination_file = (
            POSTER_FIGURE_DIRECTORY
            / f"{figure_stem}.{extension}"
        )

        if source_file.exists():
            shutil.copy2(source_file, destination_file)
        else:
            print(
                f"[WARNING] Poster figure source missing: "
                f"{source_file.name}"
            )

print(
    "[SAVE] Poster manifest figures copied consistently "
    "to the poster_figures directory."
)

# Part H — Final validation and export

## Step 22 — Notebook 08 validation

In [ ]:
generated_table_files = list(TABLE_DIRECTORY.glob("*.csv"))
generated_png_files = list(FIGURE_DIRECTORY.glob("*.png"))
generated_pdf_files = list(FIGURE_DIRECTORY.glob("*.pdf"))
generated_svg_files = list(FIGURE_DIRECTORY.glob("*.svg"))

required_corrected_outputs = [
    FIGURE_DIRECTORY
    / "figure_06_false_negative_root_causes.png",

    FIGURE_DIRECTORY
    / "figure_14_per_category_exact_span_category_f1.png",

    FIGURE_DIRECTORY
    / "figure_15_category_support_vs_f1.png",

    FIGURE_DIRECTORY
    / "figure_17_observed_errors_by_dimension_non_additive.png",

    TABLE_DIRECTORY
    / "table_07_assertion_research_summary.csv",

    TABLE_DIRECTORY
    / "table_08_temporality_research_summary.csv",

    TABLE_DIRECTORY
    / "table_09_per_category_exact_span_category_metrics.csv",
]

poster_manifest_stems = [
    "figure_02_overall_f1_comparison",
    "figure_14_per_category_exact_span_category_f1",
    "figure_16_primary_error_taxonomy",
    "figure_06_false_negative_root_causes",
    "figure_11_assertion_confusion_matrix_normalised",
    "figure_13_temporality_confusion_matrix_normalised",
]

poster_manifest_complete = all(
    (
        POSTER_FIGURE_DIRECTORY
        / f"{stem}.png"
    ).exists()
    for stem in poster_manifest_stems
)

notebook_08_validation_checks = {
    "notebook_07_validation_passed":
        failed_checks_df.empty,

    "corpus_summary_generated":
        not corpus_summary_df.empty,

    "overall_performance_generated":
        (
            len(performance_summary_df) == 4
            and performance_summary_df["F1"].notna().all()
        ),

    "false_positive_count_preserved":
        len(false_positive_df)
        == int(
            span_report.get(
                "false_positives",
                len(false_positive_df),
            )
        ),

    "false_negative_count_preserved":
        len(false_negative_df)
        == int(
            span_report.get(
                "false_negatives",
                len(false_negative_df),
            )
        ),

    "per_category_metrics_loaded":
        not per_category_df.empty,

    "per_category_support_identified":
        support_column is not None,

    "error_taxonomy_generated":
        not error_taxonomy_df.empty,

    "assertion_support_aware_metrics_generated":
        not assertion_research_summary_df.empty,

    "temporality_support_aware_metrics_generated":
        not temporality_research_summary_df.empty,

    "all_corrected_outputs_generated":
        all(path.exists() for path in required_corrected_outputs),

    "poster_manifest_files_present":
        poster_manifest_complete,

    "tables_exported":
        len(generated_table_files) >= 35,

    "png_figures_exported":
        len(generated_png_files) >= 17,

    "pdf_figures_exported":
        len(generated_pdf_files) >= 17,

    "svg_figures_exported":
        len(generated_svg_files) >= 17,
}

notebook_08_validation_df = pd.DataFrame(
    [
        {
            "Check": check,
            "Passed": bool(result),
        }
        for check, result
        in notebook_08_validation_checks.items()
    ]
)

display(notebook_08_validation_df)

notebook_08_validation_df.to_csv(
    REPORT_DIRECTORY
    / "notebook_08_final_validation.csv",
    index=False,
)

failed_notebook_08_checks = (
    notebook_08_validation_df.loc[
        ~notebook_08_validation_df["Passed"]
    ]
)

if not failed_notebook_08_checks.empty:
    print(
        "\nWARNING: Some Notebook 08 validation checks did not pass."
    )
    display(failed_notebook_08_checks)
else:
    print("\nAll corrected Notebook 08 validation checks passed.")

## Step 23 — Final machine-readable report

In [ ]:
notebook_08_report = {
    "notebook": "08-corrected",
    "title":
        "Corrected Error Analysis, Interpretation, and Research Outputs",

    "completed_at_utc":
        datetime.now(timezone.utc).isoformat(),

    "input_directory":
        str(EVALUATION_DIRECTORY),

    "output_directory":
        str(ANALYSIS_DIRECTORY),

    "corpus":
        corpus_report,

    "performance":
        performance_summary_df.to_dict(
            orient="records"
        ),

    "error_counts": {
        "exact_span_false_positives":
            int(len(false_positive_df)),
        "exact_span_false_negatives":
            int(len(false_negative_df)),
        "boundary_overlap_pairs":
            int(len(boundary_df)),
        "category_errors":
            int(len(category_errors_df)),
        "assertion_errors":
            int(len(assertion_errors_df)),
        "temporality_errors":
            int(len(temporality_errors_df)),
    },

    "primary_error_taxonomy":
        error_taxonomy_df.to_dict(
            orient="records"
        ),

    "generated_outputs": {
        "csv_tables":
            int(len(list(TABLE_DIRECTORY.glob("*.csv")))),
        "png_figures":
            int(len(list(FIGURE_DIRECTORY.glob("*.png")))),
        "pdf_figures":
            int(len(list(FIGURE_DIRECTORY.glob("*.pdf")))),
        "svg_figures":
            int(len(list(FIGURE_DIRECTORY.glob("*.svg")))),
    },

    "validation": {
        row["Check"]: bool(row["Passed"])
        for _, row
        in notebook_08_validation_df.iterrows()
    },
}

report_path = (
    REPORT_DIRECTORY
    / "notebook_08_error_analysis_report.json"
)

with open(
    report_path,
    "w",
    encoding="utf-8",
) as output_file:
    json.dump(
        notebook_08_report,
        output_file,
        indent=2,
        ensure_ascii=False,
        default=str,
    )

print("=" * 80)
print("NOTEBOOK 08 — ERROR ANALYSIS COMPLETE")
print("=" * 80)

print(f"\nTables directory : {TABLE_DIRECTORY}")
print(f"Figures directory: {FIGURE_DIRECTORY}")
print(f"Poster figures   : {POSTER_FIGURE_DIRECTORY}")
print(f"Paper figures    : {PAPER_FIGURE_DIRECTORY}")
print(f"Final report     : {report_path}")

print(
    f"\nGenerated CSV tables: "
    f"{len(list(TABLE_DIRECTORY.glob('*.csv')))}"
)

print(
    f"Generated PNG figures: "
    f"{len(list(FIGURE_DIRECTORY.glob('*.png')))}"
)

print(
    f"Generated PDF figures: "
    f"{len(list(FIGURE_DIRECTORY.glob('*.pdf')))}"
)

print(
    f"Generated SVG figures: "
    f"{len(list(FIGURE_DIRECTORY.glob('*.svg')))}"
)